# Notebook 8: Citation Dynamics
We examine how citations differ **over time** between sharing status (*sharing vs. non-sharing*) and across sharing granularities (*fix vs. trial vs. participant*).<br>
To do so, we calculate the number of citations in each calendar year post-publication for each article - ignoring "year 0" and the current year (2026) due to partial year effects. We then examine the differences in year-by-year citations across sharing categories while accounting for other article-level covariates (e.g., U.S. affiliation, venue impact score, etc.). We fit a single mixed-effects model regressing *Citations in year `k` post publication* on the article covariates, including a `C(Is Sharing) * k` interaction term to test whether the citations differ between *sharing* and *non-sharing* over time (i.e. the citation **dynamics**). We also fit a second mixed-effects model with a `C(Sharing Class) * k` interaction term to test whether the citation dynamics differ across sharing granularities.
<br><br>
**Notes:**
1. While `k` is an *ordinal* variable, we treat it as a *continuous* variable (not a *categorical* one) in the mixed-effects model to reduce the number of parameters and increase statistical power. This means the interaction term directly gives us the difference in slope between sharing and non-sharing articles (or between sharing granularities) over time.
2. We estimate the number of citations *within* year `k` (flow) and not the cumulative citations *up to* year `k` (stock). This is because the cumulative citations are highly correlated across years, which violates the independence assumption of the mixed-effects model. The flow of citations within each year is more appropriate for this analysis.
3. Because citation *flow* is zero-heavy (many articles are not cited in a given year), we need to compare different model families - Gaussian (regular), Poisson, and Negative Binomial - to determine which is the best fit for the data. We will report the best-fitting model in the manuscript, mentioning the other models' results are available online.
4. Since we use `k` as a continuous variable, we should consider non-linear effects of `k` (e.g. $log(k)$ if we beleive citations saturate/decay over time, or $k^2$ if we believe citations accelerate over time). We will compare the linear vs. non-linear models and report the best-fitting model in the manuscript.

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure

pio.renderers.default = "browser"

## Setup

In [2]:
def citations_per_year(
        articles: pd.DataFrame, grouping_col: str, cumulative: bool = False
) -> pd.DataFrame:
    """Citations per year since publication, indexed by (article, group).

    The reshape from absolute calendar years to years-since-publication is done by
    `dataset.citations_since_publication()`. This wrapper only applies the two analysis choices
    belonging to this notebook, which that helper deliberately leaves to its callers: drop the
    publication year itself (`year_0`, partial for every article), and drop any calendar year that
    has not fully elapsed for the whole sample.

    The incomplete-year cutoff comes from `dataset.citations_through_shared_year()` rather than a
    hard-coded year, so it tracks the data instead of going stale at the next re-fetch.

    :param articles: frame carrying the `Citations20XX` columns, `PublicationYear`, `LastUpdate`
        and `grouping_col`.
    :param grouping_col: column to append as the second index level, e.g. `is_sharing_data`.
    :param cumulative: True for cumulative citations (stock), False for per-year counts (flow).
    :return: frame indexed by (article, `grouping_col`), one column per year offset.
    """
    _, shared_year = dataset.citations_through_shared_year(articles)
    incomplete = [
        col for col in articles.columns
        if col.startswith("Citations") and col.removeprefix("Citations").isdigit()
        and int(col.removeprefix("Citations")) > shared_year
    ]
    per_year = dataset.citations_since_publication(articles.drop(columns=incomplete))
    per_year = per_year.drop(columns="year_0")
    per_year.columns = [int(col.removeprefix("year_")) for col in per_year.columns]
    if cumulative:
        per_year = per_year.cumsum(axis=1)
    return per_year.set_index(articles[grouping_col], append=True)

In [3]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


In [6]:
FEATURES_DF

,Is Sharing Data,Sharing Class,Has US Author,Is Open Access,Has Preprint,log(Weeks Since Pub.),log(Number of Authors),Venue Impact,Venue Name
0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision
2,0,NONE,0,0,0,5.914661,1.609438,1.419355,Visual Cognition
3,1,FIXATION,0,1,0,5.481234,1.098612,0.429831,Journal of Vision
4,1,FIXATION,0,1,0,5.723585,1.098612,3.520000,Journal of Experimental Psychology Human Perce...
5,1,TRIAL,0,1,1,5.459586,1.386294,3.520000,Journal of Experimental Psychology Human Perce...
...,...,...,...,...,...,...,...,...,...
247,0,NONE,1,1,0,6.156071,1.609438,1.542299,Quarterly Journal of Experimental Psychology
248,0,NONE,1,1,0,5.262690,1.098612,2.024390,Vision Research
249,0,NONE,0,1,0,5.292587,1.791759,3.611215,Frontiers in Psychology
250,0,NONE,1,1,0,6.134018,0.693147,0.429831,Journal of Vision


## Sharing vs. Non-Sharing

In [19]:
GROUPING_VARIABLE = "is_sharing_data"
per_year = (
    citations_per_year(combined, grouping_col=GROUPING_VARIABLE, cumulative=False)
    .droplevel(GROUPING_VARIABLE)
    .stack()
    .rename("citations")
)
per_year.index.names = [None, 'k']      # k is the year offset since publication

# todo: merge with the article-level features to include covariates in the model

---

## Sharing Granularity

In [ ]:
GROUPING_VARIABLE = "sharing_class"
# TODO

In [4]:
# TODO: drop this to see the (deprecated) residuals analysis
1/0

ZeroDivisionError: division by zero

---

## Citation Dynamics :: Univariate Comparisons (SUPERSEDED)
**This section is superseded and does not appear in the revised manuscript.** It is the analysis
reported in the submitted version, kept so the published v1 results stay reproducible. Everything
from this heading to the end of the notebook belongs to it, including the per-group comparisons at
the bottom.
<br><br>
We explor how citations differ **over time** between sharing and non-sharing articles; and whether different types of data sharing are associated with different citation dynamics.<br>
To do so, we calculate the number of citations in each calendar year post-publication for each article - ignoring "year 0" and the current year (2026) due to partial year effects. We then examine the differences in year-by-year citations across sharing categories (*sharing vs. non-sharing* and *sharing granularities*). We use a set of Mann-Whitney U-tests to compare the distributions of citations per year between groups, and correct for multiple comparisons using the Benjamini-Hochberg FDR procedure.

### (1) Sharing vs. Non-Sharing Articles
We compare the cumulative citation counts per year between articles that share data vs. those that do not.

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=True
)
# drop years with fewer than 20 articles combined
cit_dynamics_sharing = cit_dynamics_sharing.loc[:, cit_dynamics_sharing.count(axis=0) >= 20]

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    non_sharing_is_normal = stats.shapiro(non_sharing_vals).pvalue > 0.05
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    sharing_is_normal = stats.shapiro(sharing_vals).pvalue > 0.05
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    yearly_ttest = pg.ttest(non_sharing_vals, sharing_vals, paired=False, alternative="less", correction=True)
    yearly_mwtest = pg.mwu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = yearly_mwtest.loc["MWU", "U_val"]
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_is_normal": non_sharing_is_normal,
        "non_sharing_median": non_sharing_vals.median(),
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),

        "sharing_count": n2,
        "sharing_is_normal": sharing_is_normal,
        "sharing_median": sharing_vals.median(),
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),

        "t_stat": yearly_ttest.loc["T_test", "T"],
        "t_dof": yearly_ttest.loc["T_test", "dof"],
        "t_p_raw": yearly_ttest.loc["T_test", "p_val"],
        "t_cohen_d": yearly_ttest.loc["T_test", "cohen_d"],

        "U_nonshare": U_nonshare,
        "U_share": U_share,
        "mw_p_raw": yearly_mwtest.loc["MWU", "p_val"],
        "r_rb": abs(yearly_mwtest.loc["MWU", "RBC"]),
        "cles": yearly_mwtest.loc["MWU", "CLES"],
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
t_reject, t_p_corrected, _, _ = multipletests(year_stats['t_p_raw'], alpha=0.05, method='fdr_bh')
mw_reject, mw_p_corrected, _, _ = multipletests(year_stats['mw_p_raw'], alpha=0.05, method='fdr_bh')
year_stats['t_p_corrected'] = t_p_corrected
year_stats['t_is_significant'] = t_reject
year_stats['mw_p_corrected'] = mw_p_corrected
year_stats['mw_is_significant'] = mw_reject
column_order = [
    "year",
    "non_sharing_is_normal", "sharing_is_normal",
    "t_is_significant", "t_p_corrected",
    "mw_is_significant", "mw_p_corrected",
    "t_stat", "t_dof", "t_p_raw", "t_cohen_d",
    "U_share", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem", "non_sharing_median",
    "sharing_count", "sharing_mean", "sharing_sem", "sharing_median",
]
year_stats = year_stats[column_order]

year_stats

In [ ]:
LINE_ANNOTATION_X = 3

cit_dynamics_sharing_fig = go.Figure()
for is_share in cit_dynamics_sharing.index.get_level_values(GROUPING_COL).unique().sort_values():
    subset = cit_dynamics_sharing.xs(is_share, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    label = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[label.upper()]
    cit_dynamics_sharing_fig.add_trace(go.Scatter(
        name=label, legendgroup=label,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=color,
            # size=np.sqrt(count_per_year.values),
        ),
        mode='lines+markers',
    ))
    # annotate the lines
    if is_share:
        ann_y = mean_per_year[LINE_ANNOTATION_X] + 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "right"
        yanchor = "bottom"
    else:
        ann_y = mean_per_year[LINE_ANNOTATION_X] - 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "left"
        yanchor = "top"
    cit_dynamics_sharing_fig.add_annotation(
        x=LINE_ANNOTATION_X, xanchor=xanchor, xref="x",
        y=ann_y, yanchor=yanchor, yref="y",
        text=label, font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )

# add significance asterisks
SIGNIFICANCE_ANNOTATION_Y = 40
for _, row in year_stats.iterrows():
    if row["mw_p_corrected"] >= 0.1:
        continue
    fontsize = 22
    yanchor = "middle"
    if row["mw_p_corrected"] < 0.001:
        ann_text = "<b>***</b>"
    elif row["mw_p_corrected"] < 0.01:
        ann_text = "<b>**</b>"
    elif row["mw_p_corrected"] < 0.05:
        ann_text = "<b>*</b>"
    else:
        ann_text = "\u2020"     # dagger for p < 0.
        fontsize = 16
        yanchor = "bottom"
    cit_dynamics_sharing_fig.add_annotation(
        x=row["year"], xanchor="center", xref="x",
        y=SIGNIFICANCE_ANNOTATION_Y, yanchor=yanchor, yref="y",
        text=ann_text, font={**TITLE_FONT, "size": fontsize},
        showarrow=False,
    )

cit_dynamics_sharing_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Sharing vs. Non-Sharing Articles",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=1.0, xanchor="right", y=0.4, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
        visible=False,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_fig, "citation_dynamics_sharing_vs_nonsharing.png", width=800, height=400)

##### Source of the Difference
So we found a difference in yearly cumulative citations between sharing and non-sharing articles.<br>
But where does this difference come from? Is it because the sharing articles get cited more frequently in each year (i.e., higher "velocity" of citations)? Or is it because they got an initian citation boost which then accumulates over time (i.e., higher "stock" of citations)?<br>
To answer this question, we repeat the above analysis but now looking at **non-cumulative** citation counts per year (i.e., the "velocity" of citations).

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=False
)

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    year_pair_test = stats.mannwhitneyu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = year_pair_test.statistic
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),
        "sharing_count": n2,
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),
        "U_nonshare": U_nonshare,
        "p_raw": year_pair_test.pvalue,
        "r_rb": rank_biserial_corr,
        "cles": cles,
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "year",
    "is_significant", "p_corrected", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem",
    "sharing_count", "sharing_mean", "sharing_sem",
    "p_raw"
]
year_stats = year_stats[column_order]

year_stats

### (2) Different Types of Data Sharing

In [ ]:
GROUPING_COL = "data_sharing_class"

cit_dynamics_sharing_class = citations_per_year(
    combined.loc[combined["is_sharing_data"]], grouping_col=GROUPING_COL
)

year_stats = dict()
for year in cit_dynamics_sharing_class.columns:
    year_stats[year] = dict()
    class_values = []
    for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
        subset = (cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)[year]).dropna()
        if subset.empty:
            continue
        year_stats[year][f"{share_class}_count"] = subset.count()
        year_stats[year][f"{share_class}_mean"] = subset.mean()
        year_stats[year][f"{share_class}_sem"] = subset.sem()
        class_values.append(subset)
    sharegroup_test = stats.kruskal(*class_values)
    year_stats[year]["W"] = sharegroup_test.statistic
    year_stats[year]["p_raw"] = sharegroup_test.pvalue

year_stats = pd.DataFrame.from_dict(year_stats, orient='index')
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "is_significant", "p_corrected", "W", "p_raw",
    *[f"{share_class}_{stat}" for share_class in ["FIXATION", "TRIAL", "PARTICIPANT"] for stat in ["count", "mean", "sem"]]
]
year_stats = year_stats[column_order]
year_stats

In [ ]:
cit_dynamics_sharing_class_fig = go.Figure()
for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
    subset = cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    cit_dynamics_sharing_class_fig.add_trace(go.Scatter(
        name=share_class, legendgroup=share_class,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=CLASS_COLORS[share_class],
            size=count_per_year.values
        ),
        mode='lines+markers',
    ))

cit_dynamics_sharing_class_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Different Types of Data Sharing",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_class_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_class_fig, "citation_dynamics_sharing_class.png", width=800, height=400)